# Sabatini & Kaufman (2024) on DANDI:000070: prediction, encoding and decoding (notebook 2 of 2)

Paper: doi:10.1038/s41467-024-51308-7; dataset DANDI:000070 version 0.261122.2046 (the most recent
published version; the paper cites none), sessions `sub-Jenkins_ses-20090918` and
`sub-Nitschke_ses-20100923` as named in the paper's Data availability statement. The claims file was written
from the version of record and its supplement. Notebook 1 holds the data check and the claims on the
factorization and dynamics; this one recomputes the factorization (the same cross-validation, seed and k)
and tests:

| claim | here |
|---|---|
| fig5c-rotations-predict-rotations, fig6b-location-predicts-rotations, fig6c-location-tracks-direction, fig6e-kinematics-predict-location, fig7c-ldr-encoding, suppfig7a-not-memorization, fig7c-standard-encoding-worse, fig7f-ldr-decoding, trial-curvature-decoding, suppfig8-no-output-potent-dependence | tested |
| fig7f-decoder-beats-standard | tested here, without the GPFA decoders |

In [ ]:
import time, resource, warnings
import numpy as np
import pandas as pd
from scipy import stats
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import ldr
warnings.filterwarnings("ignore", category=RuntimeWarning)
np.set_printoptions(precision=3, suppress=True, linewidth=150)
T_START = time.time()
SEED_OFFSETS = [0, 1000, 2000, 3000, 4000]
DS = ldr.build("all")
NAMES = ldr.DATASETS
MONKEYS = sorted({n.split("-")[1] for n in NAMES})
for n in NAMES:
    DS[n]["bc"] = ldr.binned_counts(DS[n]["counts"])   # 10 ms counts, spikes/s, for single-trial decoding
    del DS[n]["counts"]
K = {n: ldr.cv_tbf_k(DS[n]["Xt"], DS[n]["cond"], ldr.rng("fig3g-basis-frequencies"), folds=100)["k"] for n in NAMES}
TBF = {n: ldr.eig_transform(DS[n]["X"], K[n]) for n in NAMES}
KB = {rd: ldr.kin_basis([DS[f"M1-{m}"]["kin"] for m in MONKEYS], rd) for rd in ("svd", "skewnormal")}
PARAMS = {(n, rd): ldr.reach_params(KB[rd][0], DS[n]["kin"]) for n in NAMES for rd in KB}
print("k per dataset:", K)
print("loaded in %.0f s, peak memory %.1f GB" % (time.time() - T_START, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

In [ ]:
# The kinematic basis functions: the top two velocity singular vectors over all conditions of both monkeys
# (reading svd) and their skew-normal fits (reading skewnormal).
t = ldr.TIMES
fig, ax = plt.subplots(1, 2, figsize=(10, 2.6))
for rd in KB:
    ax[0].plot(t, KB[rd][1]); ax[1].plot(t, KB[rd][0])
ax[0].set_title("velocity basis"); ax[1].set_title("position basis"); plt.show()
raw = KB["svd"][2]["raw"]
fit = KB["skewnormal"][1]
for i in range(2):
    r2 = 1 - ((raw[:, i] - fit[:, i]) ** 2).sum() / ((raw[:, i] - raw[:, i].mean()) ** 2).sum()
    print(f"velocity singular vector {i + 1}: peak at {t[np.argmax(abs(raw[:, i]))]} ms; skew-normal fit R2 {r2:.3f}")
for n in [f"M1-{m}" for m in MONKEYS]:
    P = PARAMS[(n, "svd")]
    path = ldr.params_to_path(KB["svd"][0], P)
    ve = 1 - ((path - DS[n]["kin"]) ** 2).sum((1, 2)) / ((DS[n]["kin"] - DS[n]["kin"].mean(1, keepdims=True)) ** 2).sum((1, 2))
    print(f"{n[-1]}: reach parameters ranges {np.round(P.min(0), 1)}..{np.round(P.max(0), 1)}; the two basis functions "
          f"explain {ve.mean()*100:.1f}% of the condition-mean hand path (min {ve.min()*100:.1f}%)")

In [ ]:
def ve_rows(X, Xh):
    return ldr.var_explained(X, Xh)

def ridge_code(Xf, Y, lam):
    """Code style ridge: a row of ones appended to the predictors and penalised with them;
    Xf: (n, p), Y: (n, q); returns a predictor."""
    Z = np.c_[Xf, np.ones(len(Xf))]
    W = np.linalg.solve(Z.T @ Z + lam * np.eye(Z.shape[1]), Z.T @ Y)
    return lambda A: np.c_[A, np.ones(len(A))] @ W

def pinv_sym(K):
    """Pseudo-inverse of a symmetric matrix with MATLAB's default tolerance (max(size) * eps(norm))."""
    w, V = np.linalg.eigh(K)
    tol = max(K.shape) * np.spacing(np.abs(w).max())
    inv = np.where(np.abs(w) > tol, 1 / np.where(w == 0, 1, w), 0)
    return (V * inv) @ V.T

def kernel_map(Xtr, Ytr, kern, lam):
    """Code style kernel regression (no centring): Y = Ytr' pinv(K + lam I) k(x)."""
    Kt = kern(Xtr, Xtr)
    A = pinv_sym(Kt + lam * np.eye(len(Kt))) @ Ytr
    return lambda Z: kern(Z, Xtr) @ A

OU = lambda ls: (lambda A, B: np.exp(-np.sqrt(np.maximum(((A[:, None] - B[None]) ** 2).sum(-1), 0)) / ls))
GAUSS = lambda ls: (lambda A, B: np.exp(-((A[:, None] - B[None]) ** 2).sum(-1) / ls ** 2))   # code: exp(-(d/ls)^2)

def six_folds(claim, k=0, n=72, nf=6):
    return ldr.folds_over(n, nf, ldr.rng(claim, k))

def vec(L):
    return L.reshape(len(L), -1)

## fig5c-rotations-predict-rotations
> Each rotation's parameters on a condition predicted 43 to 94% of the variance of a different-frequency
> rotation, with one model for all conditions, above chance.

Code predictOneComponentFromAnother.m: 6 folds over conditions; TBF on the training conditions; the
predictor's vectorised loading columns reduced to the left singular vectors holding 80% of the summed
singular values, with a row of ones; ridge (lambda 0.1); held-out loadings by projecting the held-out
condition on the training basis. Truth: the held-out condition's own predicted columns times the basis.
Null (code oneComponentToAnother.m): each component of a condition's activity taken from a random condition.

In [ ]:
def rot_predict(X, k, folds):
    C = len(X)
    out = {}
    for te in folds:
        tr = np.setdiff1d(np.arange(C), te)
        tb = ldr.eig_transform(X[tr], k)
        gr = ldr.groups(tb)
        ok = ~np.isnan(tb["B"][:, 0])
        Lte = X[te] @ np.linalg.pinv(tb["raw"].T)
        for a, (fa, ia) in enumerate(gr):
            Xp = vec(tb["L"][:, :, ia])
            U, S, _ = np.linalg.svd(Xp.T, full_matrices=False)
            keep = int(np.argmax(np.cumsum(S) / S.sum() > 0.8)) + 1
            sv = U[:, :keep]
            for b, (fb, ib) in enumerate(gr):
                if a == b:
                    continue
                Y = vec(tb["L"][:, :, ib])
                f = ridge_code(Xp @ sv, Y, 0.1)
                pred = f(vec(Lte[:, :, ia]) @ sv).reshape(len(te), -1, len(ib))
                Bq = tb["B"][ok][:, ib]
                truth = Lte[:, :, ib] @ Bq.T
                ve = ldr.var_explained(truth, pred @ Bq.T)
                o = out.setdefault((a, b), np.full(C, np.nan))
                o[te] = ve
    return out, [len(i) for f, i in ldr.groups(ldr.eig_transform(X, k))]

def null_components(n, g):
    tb = TBF[n]
    gr = ldr.groups(tb)
    Xn = np.zeros_like(DS[n]["X"])
    for c in range(72):
        for f, ii in gr:
            r = g.integers(72)
            Xn[c] += tb["L"][r][:, ii] @ tb["raw"][:, ii].T
    return Xn

F5C = {}
for n in NAMES:
    real, sizes = rot_predict(DS[n]["X"], K[n], six_folds("fig5c-rotations-predict-rotations"))
    null, _ = rot_predict(null_components(n, ldr.rng("fig5c-rotations-predict-rotations", 1)), K[n],
                          six_folds("fig5c-rotations-predict-rotations", 2))
    seeds = []
    for k in SEED_OFFSETS[1:]:
        rr, _ = rot_predict(DS[n]["X"], K[n], six_folds("fig5c-rotations-predict-rotations", k))
        seeds.append(np.mean([v.mean() for (a, b), v in rr.items() if sizes[a] == 2 and sizes[b] == 2]))
    F5C[n] = dict(real=real, null=null, sizes=sizes, seeds=seeds)

In [ ]:
rows = []
for n in NAMES:
    f = F5C[n]
    for (a, b), v in f["real"].items():
        nv = f["null"][(a, b)]
        rows.append(dict(dataset=n, predictor=a, predicted=b, rotations=f["sizes"][a] == 2 and f["sizes"][b] == 2,
                         mean=v.mean(), sd=v.std(ddof=1), null=nv.mean(), p_signrank=stats.wilcoxon(v, nv).pvalue,
                         p_ranksum=stats.ranksums(v, nv).pvalue))
F5 = pd.DataFrame(rows)
print(F5.round(4).to_string(index=False))
R5 = F5[F5.rotations]
print(f"rotation pairs: means {R5['mean'].min()*100:.1f}..{R5['mean'].max()*100:.1f}%; in 38.7..100: "
      f"{R5['mean'].between(0.387, 1).sum()} of {len(R5)}; above null with signed-rank p < 0.001: "
      f"{((R5['mean'] > R5['null']) & (R5.p_signrank < 0.001)).sum()} of {len(R5)} (rank-sum: "
      f"{((R5['mean'] > R5['null']) & (R5.p_ranksum < 0.001)).sum()})")
for n in NAMES:
    print(f"{n}: mean over rotation pairs under seeds +1000..+4000: {np.round(F5C[n]['seeds'], 3)}")

**Result: reproduced.** Over the 36 ordered pairs of different rotations, the mean variance explained is 40.9% to 90.5%, all inside 38.7 to 100%, and every pair is above its null with signed-rank p < 0.001. The means are stable under the other four seeds.

## fig6b-location-predicts-rotations
> The state space location linearly predicted the rotations' parameters, explaining on average 67 to 80%
> of the population variance, above chance.

Methods: the offset column predicts the vectorised remaining columns by ridge regression (lambda 0.1), 6
folds over conditions, TBF refitted on the training folds; the observed offset column joined to the
prediction, times the basis, gives the predicted activity. The intercept is not in the Methods; the code's
appended row of ones is used. Variant (code offsetToRotations.m): 2 folds, lambda 0.01, the predictor
reduced to 90% of the summed singular values. Null (code): offsets and rotations from random conditions.

In [ ]:
def offset_predict(X, Xeval, k, folds, lam, energy=None):
    C = len(X)
    pred = np.full(Xeval.shape, np.nan)
    for te in folds:
        tr = np.setdiff1d(np.arange(C), te)
        tb = ldr.eig_transform(X[tr], k)
        off = [i for f, i in ldr.groups(tb) if len(i) == 1][0][0]
        rest = [c for c in range(k) if c != off]
        P = tb["L"][:, :, off]
        Lte = X[te] @ np.linalg.pinv(tb["raw"].T)
        if energy is not None:
            U, S, _ = np.linalg.svd(P.T, full_matrices=False)
            sv = U[:, :int(np.argmax(np.cumsum(S) / S.sum() > energy)) + 1]
        else:
            sv = np.eye(P.shape[1])
        f = ridge_code(P @ sv, vec(tb["L"][:, :, rest]), lam)
        Lh = np.zeros_like(Lte)
        Lh[:, :, off] = Lte[:, :, off]
        Lh[:, :, rest] = f(Lte[:, :, off] @ sv).reshape(len(te), -1, len(rest))
        pred[te] = Lh @ tb["B"].T
    return ldr.var_explained(Xeval, pred)

def fake_offsets(n, g):
    tb = TBF[n]
    off = [i for f, i in ldr.groups(tb) if len(i) == 1][0][0]
    rest = [c for c in range(K[n]) if c != off]
    Xf = np.zeros_like(DS[n]["X"])
    for c in range(72):
        L = np.zeros_like(tb["L"][0])
        L[:, off] = tb["L"][g.integers(72)][:, off]
        L[:, rest] = tb["L"][g.integers(72)][:, rest]
        Xf[c] = L @ tb["raw"].T
    return Xf

In [ ]:
for n in NAMES:
    X = DS[n]["X"]
    v = offset_predict(X, X, K[n], six_folds("fig6b-location-predicts-rotations"), 0.1)
    nv = offset_predict(fake_offsets(n, ldr.rng("fig6b-location-predicts-rotations", 1)), X, K[n],
                        six_folds("fig6b-location-predicts-rotations", 2), 0.1)
    vc = offset_predict(X, X, K[n], six_folds("fig6b-location-predicts-rotations", 3, nf=2), 0.01, 0.9)
    seeds = [offset_predict(X, X, K[n], six_folds("fig6b-location-predicts-rotations", k), 0.1).mean() for k in SEED_OFFSETS[1:]]
    print(f"{n}: mean {v.mean()*100:.1f}% (s.d. {v.std(ddof=1)*100:.1f}); null {nv.mean()*100:.1f}%; signed-rank p = "
          f"{stats.wilcoxon(v, nv).pvalue:.2g}; in 60.3..88: {0.603 <= v.mean() <= 0.88}; code variant (2 folds, "
          f"lambda 0.01, SVD 90%) {vc.mean()*100:.1f}%; other seeds {np.round(np.array(seeds)*100, 1)}")

**Result: reproduced.** The location predicts 71.0%, 70.0%, 83.0% and 65.4% of the population variance (inside 60.3 to 88%) against a null of 31.1% to 44.1% (signed-rank p below 1e-10). The code's variant gives 78.0% to 87.9%.

## fig6c-location-tracks-direction
> In a PCA of the state space locations across conditions, location related to the overall reach
> direction, from starting point to target, in a continuous manner.

Target angle: the direction of the condition-mean hand displacement at +850 ms (code splitTrialsAngles.m
uses the last sample of the path). The sign of a PC is arbitrary, so the correlation is also given with the
PC 1-2 angle mirrored.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
for a_, n in zip(ax, NAMES):
    tb = TBF[n]
    off = [i for f, i in ldr.groups(tb) if len(i) == 1][0][0]
    L0 = tb["L"][:, :, off]
    Lc = L0 - L0.mean(0)
    U, S, Vt = np.linalg.svd(Lc, full_matrices=False)
    pcs = U[:, :2] * S[:2]
    tang = np.arctan2(DS[n]["kin"][:, -1, 1], DS[n]["kin"][:, -1, 0])
    pang = np.arctan2(pcs[:, 1], pcs[:, 0])
    r1, p1 = ldr.circ_corr(tang, pang)
    r2, p2 = ldr.circ_corr(tang, -pang)
    frac = S[:2] ** 2 / (S ** 2).sum()
    print(f"{n}: PCs 1-2 hold {frac.sum()*100:.1f}% of the offset variance; circular correlation of target angle "
          f"with PC angle {r1:.3f} (p = {p1:.2g}), mirrored {r2:.3f} (p = {p2:.2g}); best positive {max(r1, r2):.3f}, "
          f"p = {[p1, p2][int(r2 > r1)]:.2g}")
    a_.scatter(pcs[:, 0], pcs[:, 1], c=tang, cmap="hsv", s=14); a_.set_title(n); a_.set_xlabel("PC 1"); a_.set_ylabel("PC 2")
plt.tight_layout(); plt.show()

**Result: reproduced.** The angle of each condition in the plane of the first two PCs of the location follows the target angle, with circular correlation 0.657, 0.719 and 0.630 after mirroring for M1-J, PMd-J and M1-N and 0.357 for PMd-N, all with p < 0.001.

## fig6e-kinematics-predict-location
> Four reach parameters linearly predicted 37 to 71% of the variance in state space location, and kernel
> regression 69 to 81%, both above chance.

The offset columns of the factorization of all 72 conditions are the targets; the reach parameters (both
kinematic readings) the predictors; 6 folds over conditions; ridge (lambda 0.1) with intercept; kernels
(a) Ornstein-Uhlenbeck, length scale 10^3, no penalty (the LDR encoder's) and (b) Gaussian, length scale 400,
lambda 0.1. Variance explained pooled over the held-out conditions' offset columns. Shuffle: reach
parameters permuted across conditions, 1000 permutations (standard choice).

The length scales come from the paper (10^3 for the LDR encoder, 400 for the rotation predictions), where the
reach parameters are coefficients on the kinFuncEval basis, which is not published, so their scale there is
unknown. Here they are coefficients on the integrated unit-norm velocity singular vectors (positions in mm).
The cell below prints their scale and sweeps the length scale of both kernels (reading svd, no permutations)
to show how much the outcome depends on it. A second variant refits the factorization on the training
conditions of each fold, with the held-out conditions' offset columns projected on the training basis, so
that the targets too are kept out of training.


In [ ]:
def loc_cv(P, Y, folds, model):
    pred = np.zeros_like(Y)
    for te in folds:
        tr = np.setdiff1d(np.arange(len(P)), te)
        if model == "linear":
            f = ridge_code(P[tr], Y[tr], 0.1)
        elif model == "ou":
            f = kernel_map(P[tr], Y[tr], OU(1e3), 0)
        else:
            f = kernel_map(P[tr], Y[tr], GAUSS(400), 0.1)
        pred[te] = f(P[te])
    return 1 - ((Y - pred) ** 2).sum() / ((Y - Y.mean(0)) ** 2).sum()

F6E = []
for n in NAMES:
    tb = TBF[n]
    off = [i for f, i in ldr.groups(tb) if len(i) == 1][0][0]
    Y = tb["L"][:, :, off]
    folds = six_folds("fig6e-kinematics-predict-location")
    g = ldr.rng("fig6e-kinematics-predict-location", 1)
    perms = [g.permutation(72) for _ in range(1000)]
    for rd in KB:
        P = PARAMS[(n, rd)]
        for model in ("linear", "ou", "gauss"):
            v = loc_cv(P, Y, folds, model)
            nulls = np.array([loc_cv(P[p], Y, folds, model) for p in perms])
            seeds = [loc_cv(P, Y, six_folds("fig6e-kinematics-predict-location", k), model) for k in SEED_OFFSETS[1:]]
            F6E.append(dict(dataset=n, kin=rd, model=model, ve=v, null_max=nulls.max(), null_mean=nulls.mean(),
                            p=(1 + (nulls >= v).sum()) / 1001, seeds=np.round(seeds, 3)))
# scale of the predictors against the kernels' length scales
for n in NAMES[::2]:
    P = PARAMS[(n, "svd")]
    dd = np.sqrt(((P[:, None] - P[None]) ** 2).sum(-1))[np.triu_indices(72, 1)]
    print(f"monkey {n[-1]} (svd): reach parameters {np.round(P.min(0), 1)}..{np.round(P.max(0), 1)}; distance between "
          f"conditions median {np.median(dd):.1f}, max {dd.max():.1f}")
SWEEP = []
for n in NAMES:
    tb = TBF[n]
    off = [i for f, i in ldr.groups(tb) if len(i) == 1][0][0]
    Y = tb["L"][:, :, off]
    folds = six_folds("fig6e-kinematics-predict-location")
    P = PARAMS[(n, "svd")]
    for kname, ls_list in (("ou", (10, 30, 100, 300, 1e3, 1e4)), ("gauss", (10, 40, 100, 400, 4e3))):
        for ls in ls_list:
            pred = np.zeros_like(Y)
            for te in folds:
                tr = np.setdiff1d(np.arange(72), te)
                f = kernel_map(P[tr], Y[tr], OU(ls), 0) if kname == "ou" else kernel_map(P[tr], Y[tr], GAUSS(ls), 0.1)
                pred[te] = f(P[te])
            SWEEP.append(dict(dataset=n, kernel=kname, ls=ls, ve=1 - ((Y - pred) ** 2).sum() / ((Y - Y.mean(0)) ** 2).sum()))

def loc_cv_refit(n, P, folds, model):
    """Targets refitted per fold: the TBF of the training conditions; held-out offset columns from the held-out
    conditions' activity projected on the training basis."""
    X = DS[n]["X"]
    res, tot = 0.0, []
    trues, preds = [], []
    for te in folds:
        tr = np.setdiff1d(np.arange(72), te)
        tb = ldr.eig_transform(X[tr], K[n])
        off = [i for f, i in ldr.groups(tb) if len(i) == 1][0][0]
        Ytr = tb["L"][:, :, off]
        Yte = (X[te] @ np.linalg.pinv(tb["raw"].T))[:, :, off]
        if model == "linear":
            f = ridge_code(P[tr], Ytr, 0.1)
        elif model == "ou":
            f = kernel_map(P[tr], Ytr, OU(1e3), 0)
        else:
            f = kernel_map(P[tr], Ytr, GAUSS(400), 0.1)
        trues.append(Yte); preds.append(f(P[te]))
    Yt, Yp = np.concatenate(trues), np.concatenate(preds)
    return 1 - ((Yt - Yp) ** 2).sum() / ((Yt - Yt.mean(0)) ** 2).sum()
REFIT = [dict(dataset=n, kin=rd, model=model,
              ve=loc_cv_refit(n, PARAMS[(n, rd)], six_folds("fig6e-kinematics-predict-location"), model))
         for n in NAMES for rd in KB for model in ("linear", "ou", "gauss")]


In [ ]:
F6 = pd.DataFrame(F6E)
print(F6.round(3).to_string(index=False))
for rd in KB:
    sub = F6[F6.kin == rd].set_index(["dataset", "model"])
    lin = sub.xs("linear", level="model")
    for km in ("ou", "gauss"):
        ker = sub.xs(km, level="model")
        ok = lin.ve.between(0.333, 0.781).all() and ker.ve.between(0.621, 0.891).all() and \
            (lin.ve > lin.null_max).all() and (ker.ve > ker.null_max).all()
        print(f"kinematics {rd}, kernel {km}: linear in 33.3..78.1 {lin.ve.between(0.333, 0.781).sum()}/4, kernel in "
              f"62.1..89.1 {ker.ve.between(0.621, 0.891).sum()}/4, above all permutations {(lin.ve > lin.null_max).sum()}/4 "
              f"and {(ker.ve > ker.null_max).sum()}/4; all hold: {ok}")
SW = pd.DataFrame(SWEEP).pivot_table(index=["kernel", "ls"], columns="dataset", values="ve")[NAMES]
print("length-scale sweep (svd), variance explained of the location:")
print(SW.round(3).to_string())
for kname, lo, hi in (("ou", 0.621, 0.891), ("gauss", 0.621, 0.891)):
    sub = SW.loc[kname]
    print(f"  {kname}: datasets in 62.1..89.1 by length scale {dict((float(ls), int(r.between(lo, hi).sum())) for ls, r in sub.iterrows())}")
RF = pd.DataFrame(REFIT).pivot_table(index=["kin", "model"], columns="dataset", values="ve")[NAMES]
print("targets refitted within the folds:")
print(RF.round(3).to_string())
for rd in KB:
    lin = RF.loc[(rd, "linear")]
    for km in ("ou", "gauss"):
        ker = RF.loc[(rd, km)]
        print(f"  {rd}, {km}: linear in 33.3..78.1 {lin.between(0.333, 0.781).sum()}/4, kernel in 62.1..89.1 "
              f"{ker.between(0.621, 0.891).sum()}/4")


**Result: partially reproduced.** With the reading that fits best (Ornstein-Uhlenbeck kernel, either kinematic basis) the linear model explains 53.7%, 48.1%, 65.7% and 23.9% of the location (3 of 4 in 33.3 to 78.1%) and the kernel model 69.5%, 53.7%, 85.4% and 43.9% (2 of 4 in 62.1 to 89.1%); all are above all 1000 permutations. The Gaussian kernel gives 23.5% to 59.8% (0 of 4). The reach parameters lie within about -39 to 30, with a median distance of 41 between conditions; the OU kernel puts 2 of 4 in range at every length scale from 100 to 10^4 (fewer below) and the Gaussian kernel at most 1 of 4 (length scales 40 and 100), so the shortfall is not a matter of the length scale. Refitting the factorization within the folds gives the same counts (linear 3 of 4, OU 2 of 4, Gaussian 0 of 4).


## fig7c-ldr-encoding
> Predicting each condition's loading matrix from four reach parameters and multiplying by the shared
> basis functions explained 60 to 78% of the population variance with a linear model and 71 to 87% with
> kernel regression, above chance.

Code kinToTBF.m and predictDataUsingTBFandKinCondStructure(Nonlinear).m: 6 folds over conditions, TBF on the
training conditions; ridge (lambda 0.1, row of ones penalised) from the four parameters to the vectorised
loadings; kernel regression with the Ornstein-Uhlenbeck kernel, length scale 10^3, no penalty; predicted
loadings times the basis; variance explained per held-out condition. Null: the neural data permuted across
conditions, the kinematics kept (code), scored against the real data; signed-rank test.

In [ ]:
def ldr_encode(Xfit, Xeval, P, k, folds, model, nn=False, ls=1e3):
    pred = np.full(Xeval.shape, np.nan)
    for te in folds:
        tr = np.setdiff1d(np.arange(len(Xfit)), te)
        tb = ldr.eig_transform(Xfit[tr], k)
        Y = vec(tb["L"])
        if nn:
            d = ((P[te][:, None] - P[tr][None]) ** 2).sum(-1)
            Lh = Y[d.argmin(1)]
        elif model == "linear":
            Lh = ridge_code(P[tr], Y, 0.1)(P[te])
        else:
            Lh = kernel_map(P[tr], Y, OU(ls), 0)(P[te])
        pred[te] = Lh.reshape(len(te), -1, k) @ tb["B"].T
    return ldr.var_explained(Xeval, pred)

ENC = {}
for n in NAMES:
    X = DS[n]["X"]
    folds = six_folds("fig7c-ldr-encoding")
    perm = ldr.rng("fig7c-ldr-encoding", 1).permutation(72)
    for rd in KB:
        P = PARAMS[(n, rd)]
        for model in ("linear", "kernel"):
            v = ldr_encode(X, X, P, K[n], folds, model)
            nv = ldr_encode(X[perm], X, P, K[n], folds, model)
            seeds = [ldr_encode(X, X, P, K[n], six_folds("fig7c-ldr-encoding", k), model).mean() for k in SEED_OFFSETS[1:]]
            ENC[(n, rd, model)] = dict(v=v, null=nv, seeds=seeds)

In [ ]:
rows = []
for (n, rd, model), e in ENC.items():
    rows.append(dict(dataset=n, kin=rd, model=model, mean=e["v"].mean(), sd=e["v"].std(ddof=1), null=e["null"].mean(),
                     p=stats.wilcoxon(e["v"], e["null"]).pvalue, seeds=np.round(e["seeds"], 3)))
EN = pd.DataFrame(rows)
print(EN.round(4).to_string(index=False))
for rd in KB:
    lin = EN[(EN.kin == rd) & (EN.model == "linear")]
    ker = EN[(EN.kin == rd) & (EN.model == "kernel")]
    print(f"kinematics {rd}: linear in 54..85.8 {lin['mean'].between(0.54, 0.858).sum()}/4 with p < 0.001 and above null "
          f"{((lin.p < 0.001) & (lin['mean'] > lin.null)).sum()}/4; kernel in 63.9..95.7 {ker['mean'].between(0.639, 0.957).sum()}/4 "
          f"with p < 0.01 and above null {((ker.p < 0.01) & (ker['mean'] > ker.null)).sum()}/4")
# kernel length-scale sweep (reading svd, the claim's folds; no null)
ENC_LS = {}
for n in NAMES:
    for ls in (10, 100, 1e3, 1e4):
        ENC_LS[(n, ls)] = ldr_encode(DS[n]["X"], DS[n]["X"], PARAMS[(n, "svd")], K[n], six_folds("fig7c-ldr-encoding"),
                                     "kernel", ls=ls).mean()
ELS = pd.Series(ENC_LS).unstack(0)[NAMES]
print("kernel LDR encoding (svd) by OU length scale, mean variance explained:")
print(ELS.round(3).to_string())
print("datasets in 63.9..95.7 by length scale:", {float(ls): int(r.between(0.639, 0.957).sum()) for ls, r in ELS.iterrows()})


**Result: reproduced.** Linear LDR encoding explains 77.1%, 68.9%, 85.4% and 74.0% (inside 54 to 85.8%) and kernel encoding 84.7%, 86.5%, 92.9% and 88.0% (inside 63.9 to 95.7%), against nulls of 42.2% to 62.7%, with signed-rank p < 0.0001, under both kinematic bases and the other seeds. The kernel result holds at every OU length scale from 10 to 10^4 (80.8% to 92.9%).


## suppfig7a-not-memorization
> With targets held out by sextant, kernel-regression LDR encoding outperformed nearest-neighbour
> regression, so it was not memorization of the training set.

Code splitTrialsAngles.m: conditions sorted by the angle of the path's end point, six contiguous blocks of
12; kernel LDR encoding and the same model with nearest-neighbour regression (code
predictDataUsingTBFandKinNN.m); signed-rank test over conditions.

In [ ]:
for n in NAMES:
    X = DS[n]["X"]
    ang = np.arctan2(DS[n]["kin"][:, -1, 1], DS[n]["kin"][:, -1, 0])
    order = np.argsort(ang)
    folds = [np.sort(order[i * 12:(i + 1) * 12]) for i in range(6)]
    for rd in KB:
        P = PARAMS[(n, rd)]
        vk = ldr_encode(X, X, P, K[n], folds, "kernel")
        vn = ldr_encode(X, X, P, K[n], folds, "kernel", nn=True)
        p = stats.wilcoxon(vk, vn).pvalue
        print(f"{n} ({rd}): kernel {vk.mean()*100:.1f}%, nearest neighbour {vn.mean()*100:.1f}%, signed-rank p = {p:.2g}; "
              f"kernel higher with p < 0.01: {vk.mean() > vn.mean() and p < 0.01}")

**Result: reproduced.** With sextants held out, kernel encoding (78.2% to 88.8%) beats nearest-neighbour regression (70.5% to 82.4%) in every dataset under both kinematic bases, with p from 3.4e-12 to 1.5e-4.

## fig7c-standard-encoding-worse
> Standard instantaneous encoding models (linear multi-parameter, kernel regression on position, kernel
> regression on velocity) were substantially outperformed by both linear and nonlinear LDR encoding.

Code predictDataUsingKin.m and predictDataUsingKinNonlinear.m (linearControls.m, nonlinearControls.m): 100 ms
causal lag (10 bins); linear: ones, x, y, distance, cos and sin of the position angle, velocity x and y,
speed, smoothed acceleration x and y and its magnitude, ridge lambda 0.1; kernel: OU, length scale 10^12 on
(x, y, distance) or 10^11 on smoothed (vx, vy, speed), no penalty, training samples every second bin;
6 folds over conditions; variance explained per held-out condition over the lagged window. Compared with
the LDR encoders (kinematic reading svd) by signed-rank tests over conditions.

In [ ]:
def std_features(kin, kind):
    x, y = kin[:, 0], kin[:, 1]
    if kind == "linear":
        ax_ = gaussian_filter1d(np.diff(x, 2), 2, mode="nearest", truncate=2)
        ay_ = gaussian_filter1d(np.diff(y, 2), 2, mode="nearest", truncate=2)
        F = np.c_[np.ones(len(x) - 2), x[2:], y[2:], np.hypot(x[2:], y[2:]), np.cos(np.arctan2(y[2:], x[2:])),
                  np.sin(np.arctan2(y[2:], x[2:])), np.diff(x[1:]), np.diff(y[1:]), np.hypot(np.diff(x[1:]), np.diff(y[1:])),
                  ax_, ay_, np.hypot(ax_, ay_)]
        return F[10:]
    if kind == "pos":
        return np.c_[x, y, np.hypot(x, y)][10:]
    vx = gaussian_filter1d(np.diff(x), 2, mode="reflect", truncate=10 / 2)
    vy = gaussian_filter1d(np.diff(y), 2, mode="reflect", truncate=10 / 2)
    return np.c_[vx, vy, np.hypot(vx, vy)][10:]

def std_encode(n, kind, folds):
    X = DS[n]["X"]
    kin = DS[n]["kin"]
    if kind == "linear":
        D = X[:, :, 2:-10]
    elif kind == "pos":
        D = X[:, :, :-10]
    else:
        D = X[:, :, 1:-10]
    F = np.stack([std_features(kin[c], kind) for c in range(72)])
    pred = np.zeros_like(D)
    for te in folds:
        tr = np.setdiff1d(np.arange(72), te)
        Ftr = np.concatenate(F[tr]); Dtr = np.concatenate([d.T for d in D[tr]])
        if kind == "linear":
            W = np.linalg.solve(Ftr.T @ Ftr + 0.1 * np.eye(Ftr.shape[1]), Ftr.T @ Dtr)
            for c in te:
                pred[c] = (F[c] @ W).T
        else:
            ls = 1e12 if kind == "pos" else 1e11
            f = kernel_map(Ftr[::2], Dtr[::2], OU(ls), 0)
            for c in te:
                pred[c] = f(F[c]).T
    return ldr.var_explained(D, pred)

In [ ]:
T = time.time()
rows = []
for n in NAMES:
    folds = six_folds("fig7c-standard-encoding-worse")
    lin_ldr, ker_ldr = ENC[(n, "svd", "linear")]["v"], ENC[(n, "svd", "kernel")]["v"]
    for kind in ("linear", "pos", "vel"):
        v = std_encode(n, kind, folds)
        for nm, ref in (("LDR linear", lin_ldr), ("LDR kernel", ker_ldr)):
            p = stats.wilcoxon(ref, v).pvalue
            rows.append(dict(dataset=n, standard=kind, standard_mean=v.mean(), ldr=nm, ldr_mean=ref.mean(), p=p,
                             ldr_better=ref.mean() > v.mean() and p < 0.001))
ST = pd.DataFrame(rows)
print(ST.round(4).to_string(index=False))
print(f"comparisons where LDR is higher with p < 0.001: {ST.ldr_better.sum()} of {len(ST)}; ran in {time.time() - T:.0f} s")

**Result: reproduced.** In all 24 comparisons (two LDR models, three standard models, four datasets) the LDR model is higher with p < 0.001; the standard models explain 18.6% to 72.4% and the LDR models 68.9% to 92.9%.

## fig7f-ldr-decoding
> A linear LDR-based decoder applied to single-trial spiking explained on average 84 to 92% of the variance
> in hand position over time.

Code kinFromTBFSingleTrial.m and predictKinUsingTBFandDataCondStructureSingletrial.m: two folds over
conditions; on the training half, the raw SVD basis of the trial averages and a ridge map (lambda 10, row of
ones penalised) from the vectorised loadings to the four reach parameters; single-trial loadings of the
held-out half from 10 ms spike counts (spikes/s) times the pseudo-inverse of the basis; predicted parameters
times the position basis give the hand path; variance explained per trial (code getVarExplainedKinematics.m,
negatives 0) against the trial's own hand path. Null (reported): neural data of the training conditions
permuted across conditions.

In [ ]:
def decode(n, rd, folds, lam=10, Xfit=None, permute=None, potent=None):
    d = DS[n]
    X = d["X"] if Xfit is None else Xfit
    P = PARAMS[(n, rd)]
    pos_b = KB[rd][0]
    tri_pred = np.full((len(d["cond"]), 4), np.nan)
    cond_pred = np.full((72, 4), np.nan)
    for te in folds:
        tr = np.setdiff1d(np.arange(72), te)
        Xtr = X[tr] if permute is None else X[permute[tr]]
        B, L = ldr.get_basis(Xtr, K[n])
        f = ridge_code(vec(L), P[tr], lam)
        sel = np.isin(d["cond"], te)
        Lt = d["bc"][sel] @ np.linalg.pinv(B.T) if potent is None else (potent @ d["bc"][sel]) @ np.linalg.pinv(B.T)
        tri_pred[sel] = f(vec(Lt))
        cond_pred[te] = f(vec(X[te] @ np.linalg.pinv(B.T)))
    path = ldr.params_to_path(pos_b, tri_pred)
    true = d["hand10"]
    ve = 1 - (true - path).var(1, ddof=1).sum(1) / true.var(1, ddof=1).sum(1)
    cpath = ldr.params_to_path(pos_b, cond_pred)
    cve = 1 - (d["kin"] - cpath).var(1, ddof=1).sum(1) / d["kin"].var(1, ddof=1).sum(1)
    return np.clip(ve, 0, None), tri_pred, np.clip(cve, 0, None)

DEC = {}
for n in NAMES:
    folds = six_folds("fig7f-ldr-decoding", nf=2)
    perm = ldr.rng("fig7f-ldr-decoding", 1).permutation(72)
    for rd in KB:
        ve, tp, cve = decode(n, rd, folds)
        nve, _, _ = decode(n, rd, folds, permute=perm)
        seeds = [decode(n, rd, six_folds("fig7f-ldr-decoding", k, nf=2))[0].mean() for k in SEED_OFFSETS[1:]]
        DEC[(n, rd)] = dict(ve=ve, tp=tp, null=nve, seeds=seeds, folds=folds, cve=cve)

In [ ]:
for (n, rd), e in DEC.items():
    p = stats.wilcoxon(e["ve"], e["null"]).pvalue
    print(f"{n} ({rd}): mean over {len(e['ve'])} trials {e['ve'].mean()*100:.1f}% (s.d. {e['ve'].std(ddof=1)*100:.1f}, "
          f"median {np.median(e['ve'])*100:.1f}); in 75.6..100: {e['ve'].mean() >= 0.756}; condition-shuffled null "
          f"{e['null'].mean()*100:.1f}% (signed-rank p = {p:.2g}); other seeds {np.round(np.array(e['seeds'])*100, 1)}")

**Result: reproduced.** Decoding hand position from single-trial spikes explains on average 88.9%, 88.9%, 93.3% and 89.6% of its variance over trials (reading svd; 88.6% to 93.0% for skewnormal), inside 75.6 to 100%, against 5.4% to 10.1% (svd; 5.7% to 10.4% skewnormal) with the training conditions shuffled; stable over the other seeds (87.2% to 94.5% svd, 86.8% to 94.0% skewnormal).

## trial-curvature-decoding
> The LDR decoder followed trial-by-trial variation in reach curvature within a condition (rho 0.16 to 0.41).

Code studyAroundMeanParams.m: the curvature parameters (x and y on the second basis function) of each decoded
trial and of the trial's own hand path, each minus its condition mean; Pearson correlation over all held-out
trials of a dataset.

In [ ]:
for (n, rd), e in DEC.items():
    d = DS[n]
    true = ldr.reach_params(KB[rd][0], d["hand10"])
    pred = e["tp"]
    tc, pc = true.copy(), pred.copy()
    for c in range(72):
        s = d["cond"] == c
        tc[s] -= true[s].mean(0); pc[s] -= pred[s].mean(0)
    out = [stats.pearsonr(tc[:, j], pc[:, j]) for j in (2, 3)]
    ok = all(0.144 <= r[0] <= 0.451 and r[1] < 0.001 for r in out)
    print(f"{n} ({rd}): x curvature rho {out[0][0]:.3f} (p = {out[0][1]:.2g}), y curvature rho {out[1][0]:.3f} "
          f"(p = {out[1][1]:.2g}); both in 0.144..0.451 with p < 0.001: {ok}")

**Result: partially reproduced.** Under the svd basis seven of the eight curvature correlations lie in 0.144 to 0.451 with p < 0.001 (0.193 to 0.442); the y curvature of PMd-J gives 0.092 (p = 2.2e-4). Under the skew-normal basis six of eight do.

## suppfig8-no-output-potent-dependence
> Removing dimensions that encoded hand position, hand velocity and muscle activity produced no
> substantial degradation in LDR decoding.

The dataset holds no EMG, so only the kinematic dimensions are removed (as the criterion says). Code
removePotent.m: least-squares map from the condition-averaged activity (each neuron's mean removed) to hand
x, y and their differences (each sample's mean over the four removed, as the code does); its row space is
projected out of each condition's activity (and, here, out of the single trials). Decoding with and without
the potent dimensions (code kinFromTBFNoPotent.m: condition-averaged decoding, ridge 1, 2 folds; the same
folds for both), per-condition variance explained, two-sided signed-rank test.

In [ ]:
for n in NAMES:
    d = DS[n]
    X = d["X"]
    kin = d["kin"]
    D = np.concatenate([x[:, :-1] for x in X], 1)
    Kp = np.concatenate([np.c_[k[:-1, 0], k[:-1, 1], np.diff(k[:, 0]), np.diff(k[:, 1])] for k in kin])
    Dsub = D - D.mean(1, keepdims=True)
    projs = {}
    for mean_kind in ("code", "variable"):
        # code removePotent.m: kinParams - mean(kinParams, 2), the mean over the four variables of each sample;
        # variant: each variable's mean over samples
        Ksub = (Kp - (Kp.mean(1, keepdims=True) if mean_kind == "code" else Kp.mean(0, keepdims=True))).T
        M = Ksub @ np.linalg.pinv(Dsub)
        projs[mean_kind] = np.eye(len(M[0])) - np.linalg.pinv(M) @ M
        if n == NAMES[0]:
            print(f"{n}, {mean_kind} mean: first rows of Kp (x, y, dx, dy in mm) {np.round(Kp[:3], 2).tolist()}; first columns "
                  f"of Ksub {np.round(Ksub[:, :3].T, 2).tolist()}")
    if n == NAMES[0]:
        print(f"{n}: Dsub {Dsub.shape} (neurons x samples), first rows and columns {np.round(Dsub[:2, :3], 1).tolist()}; "
              f"neuron means of Dsub over samples at most {np.abs(Dsub.mean(1)).max():.1e}")
    folds = six_folds("suppfig8-no-output-potent-dependence", nf=2)
    for rd in KB:
        _, _, with_p = decode(n, rd, folds, lam=1)
        for mean_kind, Proj in projs.items():
            Xr = np.einsum("nm,cmt->cnt", Proj, X)
            _, _, without = decode(n, rd, folds, lam=1, Xfit=Xr, potent=Proj)
            p = stats.wilcoxon(with_p, without).pvalue
            print(f"{n} ({rd}, {mean_kind} mean): with potent {with_p.mean()*100:.1f}%, without {without.mean()*100:.1f}% "
                  f"(mean drop {(with_p - without).mean()*100:.2f} points); two-sided signed-rank p = {p:.3g}; p >= 0.036: {p >= 0.036}")


**Result: not reproduced.** Without EMG, only the hand position and velocity dimensions are removed. Decoding drops by 0.29 to 0.90 points (from 96.5% to 98.4% with them to 95.7% to 98.0% without, under svd), a small drop, but the signed-rank test finds it in every dataset (p = 4.5e-10 to 3.7e-06 under svd, 2.0e-10 to 8.6e-06 over both bases), below the 0.036 of the criterion. The code removes each sample's mean over the four kinematic variables (`mean(kinParams, 2)`, where removePotent.m allocates `kinParams` as samples x 4), as printed above; removing each variable's mean instead gives drops of 0.35 to 1.06 points and p = 8.4e-11 to 2.8e-06 under svd (4.1e-06 under skewnormal).


## fig7f-decoder-beats-standard
> The LDR-based decoder outperformed every other offline decoder tested, including GPFA denoising followed by
> nonlinear decoders.

The comparison decoders of the Methods (Comparing LDR-based decoding to instantaneous decoding methods) and
the code nonlinearDecodingControlsSingleTrial.m and predictKinUsingData.m: hand position (and velocity) at
time t decoded from the single-trial rates at t - 100 ms (10 bins); rates smoothed with the 20 ms Gaussian of
the preprocessing, or with a further Gaussian of s.d. sqrt(10^2 - 2^2) bins for 100 ms (code, 101 taps,
symmetric padding); the same two folds over conditions as the LDR decoder of fig7f-ldr-decoding, so that the
same held-out trials are compared; variance explained in hand position per trial over the lagged window
(code getVarExplainedKinematics.m, negatives 0); two-sided signed-rank test against the LDR decoder's per-trial
values (code: `signrank(..., nonlinearPrediction.varExplained.pos.array)`).
- Linear: ridge (lambda 1) with a row of ones, as the code.
- Kernel, trained on trial averages: Ornstein-Uhlenbeck kernel, length scale 10^2, L2 penalty 0.1, on the
  training conditions' averages and their mean hand path (about 4,000 samples).
- Kernel, trained on single trials: the Methods say "trained on a subset of single-trial data"; the code for it
  (predictKinUsingDataNonlinear.m) is not in the repository, so the subset size is open. Readings: 4,000 random
  training samples (the size of the trial-average set) and 12,000 (10 to 13% of a training half; its solve
  takes about half a minute per fold). Assumption.
- Velocity decoders: the velocity at t is decoded the same way and integrated from the trial's position at the
  bin before the window (-260 ms, before the reach) to give the position; the code's step for this (postHoc.m)
  is not in the repository. Assumption.
- GPFA decoders: not run. No GPFA implementation is in this entry's environment, and the paper and its code give
  neither the latent dimensionality nor the bin size. The criterion makes the outcome partial when LDR beats all
  but the GPFA decoders.


In [ ]:
LAG = 10
def smooth100(x):
    s = np.sqrt(10 ** 2 - 2 ** 2)
    return gaussian_filter1d(x, s, axis=-1, mode="reflect", truncate=50 / s)

def samples(x):
    """(trials, units, 121) -> (trials * 111, units): the rates at bins 0..110, 100 ms before the targets."""
    return np.transpose(x[:, :, :-LAG], (0, 2, 1)).reshape(-1, x.shape[1])

def ou_fit(Xtr, Ytr, ls=1e2, lam=0.1, chunk=4000):
    """Kernel ridge with the Ornstein-Uhlenbeck kernel, as kernel_map (no centring) but solved by Cholesky and
    evaluated in chunks, for thousands of samples."""
    nb = (Xtr ** 2).sum(1)
    Kt = np.exp(-np.sqrt(np.maximum(nb[:, None] + nb[None] - 2 * Xtr @ Xtr.T, 0)) / ls)
    Kt[np.diag_indices_from(Kt)] += lam
    W = np.linalg.solve(Kt, Ytr)
    del Kt
    def f(Z):
        out = np.zeros((len(Z), W.shape[1]))
        for i in range(0, len(Z), chunk):
            a = Z[i:i + chunk]
            out[i:i + chunk] = np.exp(-np.sqrt(np.maximum((a ** 2).sum(1)[:, None] + nb[None] - 2 * a @ Xtr.T, 0)) / ls) @ W
        return out
    return f

def targets(path, kind):
    """Position at bins LAG..120, or the velocity there (difference from the bin before)."""
    return path[:, LAG:] if kind == "pos" else path[:, LAG:] - path[:, LAG - 1:-1]

def std_decode(n, folds, smoothing, model, kind, g=None, n_sub=None):
    d = DS[n]
    Xt = d["Xt"].astype(np.float64)
    if smoothing == 100:
        Xt = smooth100(Xt)
    hand = d["hand10"]
    pred = np.full(hand[:, LAG:].shape, np.nan)
    for te in folds:
        tr = np.setdiff1d(np.arange(72), te)
        s_tr, s_te = np.isin(d["cond"], tr), np.isin(d["cond"], te)
        if model == "kernel_avg":
            D = samples(ldr.cond_average(Xt, d["cond"])[tr])
            Y = targets(d["kin"][tr], kind).reshape(-1, 2)
            f = ou_fit(D, Y)
        else:
            D = samples(Xt[s_tr])
            Y = targets(hand[s_tr], kind).reshape(-1, 2)
            if model == "linear":
                f = ridge_code(D, Y, 1.0)
            else:
                pick = np.sort(g.choice(len(D), n_sub, replace=False))
                f = ou_fit(D[pick], Y[pick])
        pred[s_te] = f(samples(Xt[s_te])).reshape(s_te.sum(), -1, 2)
    if kind == "vel":
        pred = hand[:, LAG - 1:LAG] + np.cumsum(pred, 1)
    true = hand[:, LAG:]
    ve = 1 - (true - pred).var(1, ddof=1).sum(1) / true.var(1, ddof=1).sum(1)
    return np.clip(ve, 0, None)

T = time.time()
DECODERS = [(sm, model, kind, ns) for kind in ("pos", "vel") for sm in (20, 100)
            for model, ns in (("linear", None), ("kernel_avg", None), ("kernel_trials", 4000), ("kernel_trials", 12000))
            if not (kind == "vel" and ns == 12000)]
STD = {}
for n in NAMES:
    folds = DEC[(n, "svd")]["folds"]
    g = ldr.rng("fig7f-decoder-beats-standard")
    for dec in DECODERS:
        STD[(n,) + dec] = std_decode(n, folds, dec[0], dec[1], dec[2], g, dec[3])
    # seed check of the random training subset, the strongest competitor (100 ms, position, 4,000 samples)
    STD[(n, "seeds")] = [std_decode(n, folds, 100, "kernel_trials", "pos", ldr.rng("fig7f-decoder-beats-standard", k), 4000).mean()
                         for k in SEED_OFFSETS[1:]]
print(f"standard decoders in {time.time() - T:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")


In [ ]:
rows = []
for n in NAMES:
    for dec in DECODERS:
        v = STD[(n,) + dec]
        for rd in KB:
            ldr_ve = DEC[(n, rd)]["ve"]
            rows.append(dict(dataset=n, smoothing=dec[0], model=dec[1] + ("" if dec[3] is None else f"_{dec[3]}"),
                             target=dec[2], kin=rd, standard=v.mean(), ldr=ldr_ve.mean(),
                             p=stats.wilcoxon(ldr_ve, v).pvalue, ldr_higher=ldr_ve.mean() > v.mean()))
SD = pd.DataFrame(rows)
print(SD[SD.kin == "svd"].round(4).to_string(index=False))
for rd in KB:
    sub = SD[SD.kin == rd]
    ok = (sub.p < 0.001) & sub.ldr_higher
    print(f"kinematics {rd}: LDR higher with p < 0.001 in {ok.sum()} of {len(sub)} comparisons; best standard decoder "
          f"per dataset {sub.groupby('dataset').standard.max().round(3).to_dict()}; LDR {sub.groupby('dataset').ldr.first().round(3).to_dict()}")
for n in NAMES:
    print(f"{n}: kernel on 4,000 single-trial samples (100 ms, position) under the other four seeds "
          f"{np.round(STD[(n, 'seeds')], 3)}")


**Result: partially reproduced.** Against the 14 standard decoders run per dataset (linear, kernel on trial averages and kernel on 4,000 or 12,000 single-trial samples; position and velocity; 20 and 100 ms smoothing), LDR decoding (88.9% to 93.3% under svd) is higher with signed-rank p < 0.001 in all 56 comparisons, under either kinematic basis. The best standard decoder reaches 75.1% (M1-J), 83.7% (PMd-J), 89.4% (M1-N) and 77.9% (PMd-N). The GPFA decoders were not run, so the outcome is partial under the criterion. The 4,000-sample kernel decoder moves by at most 0.004 under the other four seeds.


## Summary

| claim | outcome |
|---|---|
| fig5c-rotations-predict-rotations | reproduced |
| fig6b-location-predicts-rotations | reproduced |
| fig6c-location-tracks-direction | reproduced |
| fig6e-kinematics-predict-location | partially reproduced |
| fig7c-ldr-encoding | reproduced |
| suppfig7a-not-memorization | reproduced |
| fig7c-standard-encoding-worse | reproduced |
| fig7f-ldr-decoding | reproduced |
| trial-curvature-decoding | partially reproduced |
| suppfig8-no-output-potent-dependence | not reproduced |
| fig7f-decoder-beats-standard | partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

Numbers and machines: the conclusions quote the outputs printed in this copy. The same code in the same locked environment printed slightly different numbers on another machine (the round-2 verifier's copy): the fig6b nulls (29.2% to 44.8% there, 31.1% to 44.1% here), the skew-normal decoding of fig7f (88.4% to 92.8% there, 88.6% to 93.0% here) and the skew-normal p-values of suppfig8. No count against a criterion and no outcome differed between the two.

How the analysis differs from the paper's: The units are unsorted threshold crossings; hand positions are taken relative to each trial's start (an assumption; the code regresses positions on the basis without an intercept); the kinematic basis is computed from the two sessions here; the decoder comparison of Figure 7f runs every decoder but GPFA, with the single-trial kernel decoders trained on a random subset of 4,000 or 12,000 samples (an assumption: the Methods say a subset and give no size) and the velocity decoders integrated from the trial's position before the reach (an assumption); Supplementary Figure 8 removes only kinematic dimensions, as the dataset has no EMG.


In [ ]:
print(f"notebook ran in {(time.time() - T_START)/60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")